# Time Series Aggregation Overview

Time-series aggregation (TSA) reduces the **temporal complexity** of a time series so a
downstream optimization model can afford to solve over it. There are two broad ways to do
this: lower the **resolution** (fewer or coarser timesteps over the same calendar), or
represent the series with a few **typical periods** (representative period-shapes carrying
occurrence weights). This overview starts from that general picture, narrows to tsam's
specific building blocks, and maps the rest of the series — every step of which is traced by
hand on a tiny six-day dataset (introduced in [Preprocessing](01_preprocessing.ipynb)).

## 1  The TSA taxonomy: two axes of common methods

[Hoffmann et al. (2020)](https://www.mdpi.com/1996-1073/13/3/641) classify common TSA methods along two independent axes, shown in the table below:

* **How** periods or timesteps are grouped — by **time position** (consecutive blocks,
  calendar position) or by **feature similarity** (grouping by value).
* **What the result is** — **resolution variation** (fewer/coarser timesteps, same calendar)
  or **typical periods** (a few representative period-shapes with occurrence weights).

A ✅ marks what tsam implements:

|                   | **Resolution variation** | **Typical periods**          |
|-------------------|--------------------------|------------------------------|
| **Time-based**    | Downsampling *(use pandas)* | Averaging ✅ *(consecutive-period blocks; full calendar time-slices not built in)* |
| **Feature-based** | Segmentation ✅          | **Clustering** ✅ *(core)*   |

tsam focuses on the **feature-based column** (clustering and segmentation) plus time-based block averaging.


*Methods that fall outside this 2x2 grid* are currently out of tsam's scope — though tsam is
open to collaborations and contributions. They include shape- and time-shift-tolerant
clustering (dynamic time warping, k-shape), dimensionality-reduction pre-processing (PCA,
autoencoders), multiple time grids per season, random period sampling, and full calendar
time-slices (which need an external assignment vector — see
[Working with typical periods](../../how-to/working_with_typical_periods.ipynb)). Plain
downsampling is a one-liner in pandas (`df.resample(rule).mean()`) and is intentionally not
duplicated. tsam also never performs *cross-sectional grouping of time series*: it preserves
the input's dimensionality, so an `N`-attribute series stays `N`-attribute throughout.

## 2  How tsam decomposes the clustering step

[Hoffmann et al. (2020)](https://www.mdpi.com/1996-1073/13/3/641) decompose feature-based time-series clustering into **five
fundamental aspects** — and tsam exposes each one as an independent choice:

> This means that time series clustering includes five fundamental aspects:
>
> - A normalization (and sometimes a dimensionality reduction).
> - A distance metric.
> - A clustering algorithm.
> - A method to choose representatives.
> - A rescaling step in the case of non-centroid based clustering algorithms.
>

Those five aspects shape both tsam's architecture and the order of this series. The key
consequence is that the **clustering algorithm** (which periods are grouped together) and the
**method to choose representatives** (how each group becomes one profile) are *separate,
recombinable* steps. Classical algorithms tie the two together — k-means uses the mean,
k-medoids the medoid — whereas tsam lets you choose each one independently.

Those recombinable axes, and the optional steps that hang off them, are the whole pipeline in
one picture:

![The aggregation pipeline: grouping x representation, plus the optional post-steps](../../assets/architecture/aggregation_methods.svg)

## 3  What tsam implements: clustering x representation

The first table below lists the clustering methods tsam implements, grouped by the
**paradigm** each belongs to and with the representation each one defaults to. The second
lists every available representation and the goal it serves.

**Axis 1 — Grouping (clustering) methods** *(which periods go together)*

All clustering methods live together under **section 2, `02_clustering/`**. The **paradigm**
column is what organizes that section: the three feature-based paradigms optimize genuinely
different objectives, and each gets its own notebook (2.1–2.3). Block averaging (2.4) is
time-based and sits outside them.

| Paradigm | Clustering method | tsam name | Default representation | Notebook |
|---|---|---|---|---|
| **Partitional** | k-means | `kmeans` | mean | [2.1](02_clustering/01_partitional_clustering.ipynb) |
| **Partitional** | k-medoids | `kmedoids` | medoid | [2.1](02_clustering/01_partitional_clustering.ipynb) |
| **Agglomerative** | Hierarchical Ward | `hierarchical` | medoid | [2.2](02_clustering/02_agglomerative_clustering.ipynb) |
| **Agglomerative** | Contiguous Ward | `contiguous` | medoid | [2.2](02_clustering/02_agglomerative_clustering.ipynb) |
| **Extremal-prototype** | k-maxoids | `kmaxoids` | maxoid | [2.3](02_clustering/03_extremal_prototype_selection.ipynb) |
| *time-based* | Block averaging | `averaging` | mean | [2.4](02_clustering/04_averaging.ipynb) |

What separates the three paradigms:

* **Partitional** and **agglomerative** both chase the within-cluster distance $J$ that most of
  this guide builds on, and both leave the representative in the **middle** of its group. They
  differ only in *how* the groups are built — iterative refinement versus bottom-up merging.
* **Extremal-prototype** selection (`kmaxoids`) is the odd one out: it maximizes the spread
  $E(M)$ *between* representatives instead, so they land on the **extremes** rather than the
  middle. It never looks at cluster membership while choosing them — the partition is a
  **by-product**, formed afterwards by assigning each period to its nearest prototype.
* **Block averaging** never looks at the values at all; it groups by calendar position.

**Axis 2 — Representation methods** *(how each group becomes one profile)*

| Representation | tsam name | Goal — what it preserves | What it produces | Notebook |
|---|---|---|---|---|
| Centroid | `mean` | the group's average | per-timestep average of the group | [03](03_representation.ipynb) |
| Medoid | `medoid` | a real, internally consistent period | the most central real period | [03](03_representation.ipynb) |
| Maxoid | `maxoid` | the group's extreme | the most extreme real period | [03](03_representation.ipynb) |
| Distribution | `distribution` | the duration curve *and* the average | values re-sorted to keep the duration curve | [03](03_representation.ipynb) |
| Distribution + min/max | `distribution_minmax` | the duration curve *and* the envelope | duration curve with per-column min/max preserved | [03](03_representation.ipynb) |
| Min/max + mean | `minmax_mean` | the average, with chosen extremes protected | mean with per-column extremes preserved | [03](03_representation.ipynb) |

The goal column is the whole reason there is a choice here. **No rule preserves everything**,
and each buys its property with a different currency: a profile a physical system could
actually have experienced (`medoid`, `maxoid`), the correct *distribution* of values for
sizing storage and peak capacity (`distribution`), or a guaranteed envelope
(`distribution_minmax`, `minmax_mean`). Choosing a representation is choosing which of these
you cannot afford to lose.

One consequence runs through the rest of the series: only `mean` and `distribution` leave a
group's **mean** intact — `mean` by definition, `distribution` because it re-uses the group's
own values in a different order. The other four leave the aggregated series claiming the wrong
total, which is exactly what the optional [rescaling](05_rescaling.ipynb) step exists to
repair.

> **Two senses of "maxoid".** The k-maxoids **clustering** method (Axis 1) *selects*
> spread-maximal periods while grouping; the maxoid **representation** (Axis 2) *picks* the most
> extreme member of an already-formed cluster. Same word, different pipeline steps — you can use
> the maxoid representation with any clustering (e.g. Ward + maxoid).

In addition to the clustering and representation steps, the following optional methods can be
used to adapt the aggregation to your needs. They are ordered here as the pipeline applies them
— extremes and rescaling in the clustering phase, segmentation last:

* **Extreme periods** (`ExtremeConfig`) — a *clustering extension* applied right after
  clustering: inject peak/trough periods into the cluster set (`append` / `replace` /
  `new_cluster`) so they survive the aggregation. See [04](04_extreme_periods.ipynb).
* **Rescaling** (`preserve_column_means`) — a *representation post-step* (the review's fifth
  aspect): restore column means after a non-centroid representation. See [05](05_rescaling.ipynb).
* **Segmentation** (`SegmentConfig`) — the *resolution-variation axis*, applied last to the
  formed typical periods: merge adjacent timesteps into fewer segments. See [06](06_segmentation.ipynb).

*Natural extension points* — methods that fit this same grouping x representation pipeline
and could be added without architectural changes: k-medians and k-centers, other
agglomerative linkages (single / complete / average), additional distance methods (L1), and
further representation methods.

## How this series is organized

Each notebook goes deep on one part of the pipeline above, all on the same tiny six-day
dataset, and in the order the pipeline applies them:

1. [Preprocessing](01_preprocessing.ipynb) — normalization and unstacking to the period matrix (and a first look at the dataset).
2. **Clustering** (`02_clustering/`) — four ways to group the periods:
    1. [Partitional clustering](02_clustering/01_partitional_clustering.ipynb) — k-means and k-medoids.
    2. [Agglomerative clustering](02_clustering/02_agglomerative_clustering.ipynb) — hierarchical and contiguous Ward.
    3. [Extremal-prototype selection](02_clustering/03_extremal_prototype_selection.ipynb) — k-maxoids, the spread-maximizing method.
    4. [Averaging](02_clustering/04_averaging.ipynb) — positional block grouping (the one time-based method).
3. [Representation](03_representation.ipynb) — mean / medoid / maxoid / distribution strategies for turning each cluster into one profile.
4. [Extreme periods](04_extreme_periods.ipynb) — append / replace / new_cluster strategies that inject peaks into the cluster set.
5. [Rescaling](05_rescaling.ipynb) — restoring totals and denormalizing to physical units.
6. [Segmentation](06_segmentation.ipynb) — fewer timesteps within each typical period.

### Further reading

* [Notation and equations](../../reference/notation.md) — every symbol and formula on one page
* [Pipeline Guide](../background/architecture/pipeline_guide.md) — the four pipeline phases
* [Clustering methods](../../how-to/clustering_methods.ipynb) — accuracy and speed benchmarks